In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score, f1_score
import warnings
warnings.filterwarnings('ignore')

In [7]:
data7 = r"C:\Users\rajeshkumar.t\Downloads\6025ac4bfe90ad7c55f5526d6e9af44a.csv"
input_df= pd.read_csv(data7)
print(input_df.columns)

Index(['oms_version', 'id', 'order_item_id', 'type', 'flow_type',
       'order_external_id', 'checkout_id', 'status', 'net_filter_flag', 'gmv',
       ...
       'brand_classification_key', 'cms_vertical', 'is_large', 'is_bulk_order',
       'order_sales_app', 'order_sales_experience', 'order_tags', 'unit_tags',
       'is_extra_saver_flag', 'order_date_key'],
      dtype='object', length=226)


In [16]:
def prepare_propensity_data(input_df):
    tier_map = {'Tier 1A': 4, 'Tier 1B': 3, 'Tier 2': 2, 'Tier 3': 1}
    input_df['city_rank'] = input_df['city_tier'].map(tier_map).fillna(0)
    input_df['is_new'] = input_df['new_customer_flag'].map({'TRUE': 1, 'FALSE': 0, True: 1, False:0}).fillna(0)
    input_df['is_premium_phone'] = (input_df['listing_price'] > 50000).astype(int)
    input_df['is_fk_assured'] = input_df['is_fk_assured'].fillna('unknown').astype(str)
    input_df['order_sales_channel'] = input_df['order_sales_channel'].fillna('unknown').astype(str)

    X_numeric = input_df[['listing_price', 'city_rank', 'is_new', 'is_premium_phone']]
    X_categorical = pd.get_dummies(input_df[['is_fk_assured', 'order_sales_channel']], drop_first=True)
    X = pd.concat([X_numeric, X_categorical], axis=1)
    y= input_df['is_extra_saver_flag']
    return X,y

X, y = prepare_propensity_data(input_df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, stratify = y, random_state=42)

model = XGBClassifier(n_estimators = 50, max_depth=4, learning_rate= 0.1, random_state=42)
model.fit(X_train, y_train)

df_test_results = X_test.copy()
df_test_results['propensity_score'] = model.predict_proba(X_test)[:,1]

auc = roc_auc_score(y_test, df_test_results['propensity_score'])
print(f"Prppensity Model AUC: {auc:.4f}")

Prppensity Model AUC: 0.8317


In [18]:
importance_df = pd.DataFrame({
    'Features': X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)
print("Top Drivers")
print(importance_df.head(10))

Top Drivers
                         Features  Importance
4              is_fk_assured_True    0.740196
0                   listing_price    0.132463
1                       city_rank    0.035024
5  order_sales_channel_MobileSite    0.033186
7      order_sales_channel_iOSApp    0.032961
2                          is_new    0.026170
3                is_premium_phone    0.000000
6         order_sales_channel_WEB    0.000000
